# Adaptive CLIC-Net Pretraining on PTB-XL

This notebook pretrains only CLIC-Net: the StandardCNNTransformer backbone plus a learnable class-specific counterfactual lead mask. No fold loop and no baseline variant are run here.


## 1. Imports and CONFIG


In [ ]:
import ast
import csv
import json
import logging
import math
import os
import random
import re
import time
import traceback
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    balanced_accuracy_score,
)
from torch.utils.data import DataLoader, Dataset

sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore', message='enable_nested_tensor is True.*')
warnings.filterwarnings('ignore', category=DeprecationWarning)

PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebook' else Path('.').resolve()
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')


def env_int(name, default):
    value = os.environ.get(name, '').strip()
    if value in {'', '0'}:
        return default
    try:
        parsed = int(value)
    except ValueError:
        return default
    return parsed if parsed > 0 else default

CONFIG = {
    'seed': env_int('CLICNET_SEED', 42),
    'dataset_dir': str(PROJECT_ROOT / 'dataset' / 'ptb_xl'),
    'dataset_variant': 'beat_signals_100hz_recordwise',
    'augmentation_enabled': False,
    'balance_train_beats': True,
    'train_beats_per_class': 900,
    'output_base_dir': str(PROJECT_ROOT / 'outputs' / '2_pretrain_clicnet'),
    'run_name': RUN_TIMESTAMP,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 2,
    'batch_size': 64,
    'epochs': env_int('CLICNET_EPOCHS', 50),
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'scheduler_patience': 5,
    'early_stopping_enabled': False,
    'early_stopping_patience': 15,
    'loss_function': 'CrossEntropyLoss',
    'classification_head': 'softmax',
    'baseline_variant': 'standard_temporal_cnn_transformer_cls',
    'pooling_type': 'cls_token',
    'model': {
        'input_leads': 12,
        'signal_length': 65,
        'stem_channels': 32,
        'cnn_channels': 64,
        'token_dim': 128,
        'num_heads': 4,
        'transformer_layers': 2,
        'cnn_blocks': 2,
        'max_sequence_tokens': 65,
        'transformer_ff_dim': 256,
        'dropout': 0.20,
        'main_outputs': 4,
        'sub_outputs': 0,
    },
}

MAIN_LABELS = ['Normal', 'Anterior', 'Inferior', 'Lateral']
MAIN_LABEL_DISPLAY = ['NORM', 'AMI', 'IMI', 'Lateral-involved MI']
MAIN_LABEL_TO_INDEX = {label: idx for idx, label in enumerate(MAIN_LABELS)}
INDEX_TO_MAIN_LABEL = {idx: label for label, idx in MAIN_LABEL_TO_INDEX.items()}
LEAD_ORDER = ['I', 'aVL', 'II', 'III', 'aVF', 'aVR', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

BASELINE_ARCHITECTURE_CONFIG = {
    'model_name': 'CLIC-Net',
    'baseline_variant': CONFIG['baseline_variant'],
    'pooling_type': CONFIG['pooling_type'],
    'uses_alpa_module': False,
    'uses_anatomical_lead_prior': False,
    'uses_class_specific_attention': False,
    'uses_attention_context': False,
    'uses_learnable_attention_pooling': False,
    'aggregation': 'CLS token pooling after a single temporal Transformer encoder',
    'lead_order': LEAD_ORDER,
    'note': 'CLIC-Net uses a conventional 12-channel CNN followed by one temporal Transformer stack and CLS-token pooling, plus a learnable counterfactual lead-intervention mask. It does not define fixed anatomical lead priors.',
}
LABEL_MAPPING = {
    'model_name': 'CLIC-Net',
    'task_type': 'single_label_4_class_softmax_counterfactual',
    'main_label_order': MAIN_LABELS,
    'main_label_display': MAIN_LABEL_DISPLAY,
    'class_to_index': MAIN_LABEL_TO_INDEX,
    'index_to_class': INDEX_TO_MAIN_LABEL,
    'target_format': 'integer class index',
    'head_activation_for_inference': 'softmax',
    'baseline_variant': CONFIG['baseline_variant'],
    'pooling_type': CONFIG['pooling_type'],
    'note': 'CLIC-Net uses a standard temporal CNN-Transformer classifier with a data-driven counterfactual lead-intervention mask. No fixed anatomical lead-prior attention is used.',
    'recommended_loss': 'CrossEntropyLoss',
}

LOSS_ROLE_MAP = {
    'CrossEntropyLoss': 'Directly optimizes mutually exclusive 4-class classification logits for Normal, Anterior, Inferior, and Lateral.',
}


CONFIG.update({
    'variant': 'clicnet_counterfactual',
    'counterfactual_margin': 0.5,
    'lambda_sens': 0.1,
    'lambda_cons': 0.05,
    'lambda_sparse': 0.01,
    'lambda_div': 0.01,
})

print('Device:', CONFIG['device'])
print('Dataset:', CONFIG['dataset_dir'])
print('Run:', CONFIG['run_name'])
print('CLIC-Net variant:', CONFIG['variant'])


## 2. Output directory and logging setup


In [ ]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
OUTPUT_DIR = Path(CONFIG['output_base_dir']) / CONFIG['run_name']
CONFIG_DIR = OUTPUT_DIR / 'configs'; LOG_DIR = OUTPUT_DIR / 'logs'; CKPT_DIR = OUTPUT_DIR / 'checkpoints'; PLOT_DIR = OUTPUT_DIR / 'plots'; PRED_DIR = OUTPUT_DIR / 'predictions'; TRANSFER_DIR = OUTPUT_DIR / 'transfer_ready'; METRICS_DIR = OUTPUT_DIR / 'metrics'; PER_5_EPOCH_CKPT_DIR = CKPT_DIR / 'per_5fold'; ROOT_METRICS_DIR = METRICS_DIR
for directory in [CONFIG_DIR, LOG_DIR, CKPT_DIR, PLOT_DIR, PRED_DIR, TRANSFER_DIR, METRICS_DIR, PER_5_EPOCH_CKPT_DIR]: directory.mkdir(parents=True, exist_ok=True)
with open(CONFIG_DIR/'config.json','w') as f: json.dump(CONFIG,f,indent=2)
with open(CONFIG_DIR/'label_mapping.json','w') as f: json.dump(LABEL_MAPPING,f,indent=2)
with open(CONFIG_DIR/'clicnet_architecture_config.json','w') as f: json.dump(BASELINE_ARCHITECTURE_CONFIG,f,indent=2)
logger=logging.getLogger('CLIC-Net pretrain logger'); logger.setLevel(logging.INFO); logger.handlers.clear()
formatter=logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
for h in [logging.StreamHandler(), logging.FileHandler(LOG_DIR/'train.log'), logging.FileHandler(PROJECT_ROOT/'output.log')]: h.setFormatter(formatter); logger.addHandler(h)
eh=logging.FileHandler(LOG_DIR/'error.log'); eh.setLevel(logging.ERROR); eh.setFormatter(formatter); logger.addHandler(eh)
RUN_STARTED_AT=datetime.now(); RUN_START_TIME=time.time()
logger.info('CLIC-Net pretrain logger | seed=%s output=%s', CONFIG['seed'], OUTPUT_DIR)


## 3. Data loading


In [ ]:
logger.info('Data loading started.')

def load_split(dataset_dir, split):
    split_dir = Path(dataset_dir) / split
    beat_signal_path = split_dir / 'x_beats.npy'
    record_signal_path = split_dir / 'x.npy'
    if beat_signal_path.exists():
        x = np.load(beat_signal_path).astype(np.float32)
    elif record_signal_path.exists():
        x = np.load(record_signal_path).astype(np.float32)
    else:
        raise FileNotFoundError(f'No signal array found in {split_dir}. Expected x_beats.npy or x.npy')

    labels_path = split_dir / 'labels.csv'
    metadata_path = split_dir / 'metadata.csv'
    if labels_path.exists():
        labels_df = pd.read_csv(labels_path)
    elif metadata_path.exists():
        labels_df = pd.read_csv(metadata_path)
    else:
        raise FileNotFoundError(f'No label/metadata CSV found in {split_dir}. Expected labels.csv or metadata.csv')

    metadata_df = pd.read_csv(metadata_path) if metadata_path.exists() else labels_df.copy()
    return x, labels_df, metadata_df

DATASET_DIR = Path(CONFIG['dataset_dir'])
x_train, labels_train, meta_train = load_split(DATASET_DIR, 'train')
x_val, labels_val, meta_val = load_split(DATASET_DIR, 'val')
x_test, labels_test, meta_test = load_split(DATASET_DIR, 'test')

lead_order_file = DATASET_DIR / 'lead_order.csv'
config_file = DATASET_DIR / 'config.json'
if lead_order_file.exists():
    lead_df = pd.read_csv(lead_order_file)
    lead_col = 'lead' if 'lead' in lead_df.columns else lead_df.columns[0]
    dataset_leads = lead_df[lead_col].tolist()
    if dataset_leads != LEAD_ORDER:
        raise ValueError(f'Lead order mismatch. Expected {LEAD_ORDER}, got {dataset_leads}')
elif config_file.exists():
    with open(config_file) as f:
        dataset_config = json.load(f)
    dataset_leads = dataset_config.get('lead_order')
    if dataset_leads and dataset_leads != LEAD_ORDER:
        raise ValueError(f'Lead order mismatch. Expected {LEAD_ORDER}, got {dataset_leads}')
else:
    logger.warning('No lead_order.csv or config.json found in dataset directory; using notebook LEAD_ORDER.')

if x_train.shape[1] != CONFIG['model']['signal_length']:
    logger.warning('Updating CONFIG model signal_length from %s to detected train length %s', CONFIG['model']['signal_length'], x_train.shape[1])
    CONFIG['model']['signal_length'] = int(x_train.shape[1])
    with open(CONFIG_DIR / 'config.json', 'w') as f:
        json.dump(CONFIG, f, indent=2)

logger.info('Data loading completed.')
logger.info('Train shape: %s | Val shape: %s | Test shape: %s', x_train.shape, x_val.shape, x_test.shape)
logger.info('Train/val/test beat samples: %d / %d / %d', len(x_train), len(x_val), len(x_test))
display(pd.read_csv(DATASET_DIR / 'split_summary.csv'))


## 4. Classification label mapping and validation


In [ ]:
logger.info('Classification label mapping started.')

MAIN_LABEL_TO_INDEX = {label: idx for idx, label in enumerate(MAIN_LABELS)}
INDEX_TO_MAIN_LABEL = {idx: label for label, idx in MAIN_LABEL_TO_INDEX.items()}


def parse_vector_column(value):
    if isinstance(value, np.ndarray):
        return value.astype(np.float32)
    if isinstance(value, list):
        return np.asarray(value, dtype=np.float32)
    if pd.isna(value):
        return np.asarray([], dtype=np.float32)
    return np.asarray(ast.literal_eval(str(value)), dtype=np.float32)


def build_class_targets(labels_df):
    labels_df = labels_df.copy().reset_index(drop=True)
    if 'main_label_name' in labels_df.columns:
        y_class = labels_df['main_label_name'].map(MAIN_LABEL_TO_INDEX).to_numpy(dtype=np.int64)
    elif 'main_label_vector' in labels_df.columns:
        main_matrix = np.vstack(labels_df['main_label_vector'].map(parse_vector_column).to_numpy()).astype(np.float32)
        y_class = np.argmax(main_matrix, axis=1).astype(np.int64)
        labels_df['main_label_name'] = [INDEX_TO_MAIN_LABEL[int(i)] for i in y_class]
    else:
        raise ValueError('Dataset must contain main_label_name or main_label_vector for strict 4-class training.')

    if np.any(pd.isna(y_class)):
        raise ValueError('Found labels outside MAIN_LABELS.')
    labels_df['class_index'] = y_class.astype(int)
    return y_class.astype(np.int64), labels_df


def validate_classification_labels(labels_df, y_class, split_name):
    if y_class.ndim != 1:
        raise ValueError(f'{split_name}: class target must be 1D, got {y_class.shape}')
    if not np.isin(y_class, np.arange(len(MAIN_LABELS))).all():
        raise ValueError(f'{split_name}: class target contains invalid class index')
    counts = pd.Series(y_class).value_counts().reindex(range(len(MAIN_LABELS)), fill_value=0)
    counts.index = MAIN_LABELS
    logger.info('%s class distribution: %s', split_name, counts.to_dict())
    print(f'=== {split_name} classification label validation ===')
    display(counts.rename('count').to_frame())
    example_cols = [col for col in ['beat_id', 'record_id', 'raw_label', 'clinical_sub_label', 'main_label_name', 'class_index'] if col in labels_df.columns]
    display(labels_df[example_cols].head(10))


y_train, labels_train = build_class_targets(labels_train)
y_val, labels_val = build_class_targets(labels_val)
y_test, labels_test = build_class_targets(labels_test)

validate_classification_labels(labels_train, y_train, 'train')
validate_classification_labels(labels_val, y_val, 'val')
validate_classification_labels(labels_test, y_test, 'test')


def downsample_train_beats_per_class(x, y, labels_df, beats_per_class, seed=42):
    rng = np.random.default_rng(seed)
    selected_indices = []
    rows = []
    for class_index, class_name in enumerate(MAIN_LABELS):
        class_indices = np.flatnonzero(y == class_index)
        if len(class_indices) == 0:
            raise ValueError(f'No training beats found for class {class_name}.')
        n_select = min(int(beats_per_class), len(class_indices))
        chosen = rng.choice(class_indices, size=n_select, replace=False)
        selected_indices.append(chosen)
        rows.append({
            'class_index': int(class_index),
            'class_name': class_name,
            'available_train_beats': int(len(class_indices)),
            'selected_train_beats': int(n_select),
        })
    selected_indices = np.concatenate(selected_indices)
    rng.shuffle(selected_indices)
    summary = pd.DataFrame(rows)
    return x[selected_indices], y[selected_indices], labels_df.iloc[selected_indices].reset_index(drop=True), summary

if CONFIG.get('balance_train_beats', False):
    x_train, y_train, labels_train, train_balance_summary = downsample_train_beats_per_class(
        x_train,
        y_train,
        labels_train,
        CONFIG['train_beats_per_class'],
        seed=CONFIG['seed'],
    )
    train_balance_summary.to_csv(CONFIG_DIR / 'train_beat_balance_summary.csv', index=False)
    logger.info('Training beats downsampled per class: %s', train_balance_summary.to_dict(orient='records'))
    print('=== balanced training beat subset ===')
    display(train_balance_summary)
    validate_classification_labels(labels_train, y_train, 'train_balanced')
else:
    train_balance_summary = None

logger.info('Classification label mapping completed.')


## 5. Dataset and DataLoader


In [ ]:
logger.info('Dataset normalization and DataLoader setup started.')

class PerLeadZScore:
    def __init__(self, eps=1e-6):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, x):
        self.mean_ = x.mean(axis=(0, 1), keepdims=True)
        self.std_ = x.std(axis=(0, 1), keepdims=True)
        self.std_ = np.maximum(self.std_, self.eps)
        return self

    def transform(self, x):
        return ((x - self.mean_) / self.std_).astype(np.float32)

normalizer = PerLeadZScore().fit(x_train)
x_train_n = normalizer.transform(x_train)
x_val_n = normalizer.transform(x_val)
x_test_n = normalizer.transform(x_test)
np.save(TRANSFER_DIR / 'normalizer_mean.npy', normalizer.mean_.astype(np.float32))
np.save(TRANSFER_DIR / 'normalizer_std.npy', normalizer.std_.astype(np.float32))


class ECGClassificationDataset(Dataset):
    def __init__(self, x, y, labels_df=None):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.labels_df = labels_df.reset_index(drop=True) if labels_df is not None else None

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {'x': self.x[idx], 'y': self.y[idx], 'idx': idx}

train_ds = ECGClassificationDataset(x_train_n, y_train, labels_train)
val_ds = ECGClassificationDataset(x_val_n, y_val, labels_val)
test_ds = ECGClassificationDataset(x_test_n, y_test, labels_test)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())

logger.info('Dataset normalization and DataLoader setup completed.')
logger.info('DataLoaders ready: train=%d val=%d test=%d', len(train_ds), len(val_ds), len(test_ds))
logger.info('DataLoader batches: train=%d val=%d test=%d', len(train_loader), len(val_loader), len(test_loader))


## 6. Standard CNN-Transformer and Adaptive CLIC-Net


In [ ]:
logger.info('CLIC-Net model definition and instantiation started.')

class ConvBNGELU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=7, stride=1, dropout=0.1):
        super().__init__()
        padding = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class BasicResidualConvBlock(nn.Module):
    """Small sequential residual CNN block used as a conventional baseline."""
    def __init__(self, channels, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))


class StandardCNNTransformerBackbone(nn.Module):
    def __init__(self, config):
        super().__init__()
        m = config['model']
        self.input_leads = m['input_leads']
        self.token_dim = m['token_dim']
        cnn_blocks = m.get('cnn_blocks', 2)
        transformer_layers = m.get('transformer_layers', m.get('cross_lead_layers', 1) + m.get('temporal_layers', 1))
        max_tokens = m.get('max_sequence_tokens', m['signal_length'])

        layers = [
            ConvBNGELU(m['input_leads'], m['stem_channels'], kernel_size=7, stride=1, dropout=m['dropout']),
            nn.MaxPool1d(kernel_size=2, stride=2),
            ConvBNGELU(m['stem_channels'], m['cnn_channels'], kernel_size=5, stride=1, dropout=m['dropout']),
            nn.MaxPool1d(kernel_size=2, stride=2),
        ]
        for _ in range(cnn_blocks):
            layers.append(BasicResidualConvBlock(m['cnn_channels'], dropout=m['dropout']))
        self.cnn_backbone = nn.Sequential(*layers)
        self.token_projection = nn.Conv1d(m['cnn_channels'], m['token_dim'], kernel_size=1, bias=False)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, m['token_dim']))
        self.positional_embedding = nn.Parameter(torch.zeros(1, max_tokens + 1, m['token_dim']))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=m['token_dim'], nhead=m['num_heads'], dim_feedforward=m['transformer_ff_dim'],
            dropout=m['dropout'], batch_first=True, activation='gelu', norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)
        self.norm = nn.LayerNorm(m['token_dim'])
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.positional_embedding, std=0.02)

    def forward(self, x):
        # Input x: [batch, time, 12 leads]. A conventional CNN treats the 12 leads as input channels.
        x = x.permute(0, 2, 1)
        features = self.cnn_backbone(x)
        tokens = self.token_projection(features).permute(0, 2, 1)
        b, seq_len, _ = tokens.shape
        cls = self.cls_token.expand(b, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        if tokens.size(1) > self.positional_embedding.size(1):
            raise ValueError(f'Sequence length {tokens.size(1)} exceeds positional embedding length {self.positional_embedding.size(1)}')
        tokens = tokens + self.positional_embedding[:, :tokens.size(1), :]
        encoded = self.transformer(tokens)
        pooled_features = self.norm(encoded[:, 0])
        return {'pooled_features': pooled_features}


class StandardCNNTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        m = config['model']
        self.backbone = StandardCNNTransformerBackbone(config)
        self.classification_head = nn.Sequential(
            nn.LayerNorm(m['token_dim']),
            nn.Linear(m['token_dim'], m['token_dim']),
            nn.GELU(),
            nn.Dropout(m['dropout']),
            nn.Linear(m['token_dim'], m['main_outputs']),
        )

    def forward(self, x):
        backbone_out = self.backbone(x)
        logits = self.classification_head(backbone_out['pooled_features'])
        return {
            'logits': logits,
            'probabilities': torch.softmax(logits, dim=1),
            'pooled_features': backbone_out['pooled_features'],
        }


def freeze_for_transfer(model, scheme):
    for param in model.parameters():
        param.requires_grad = True
    if scheme == 'frozen_backbone':
        for param in model.backbone.parameters():
            param.requires_grad = False
    elif scheme == 'partial_finetune':
        for param in model.backbone.cnn_backbone.parameters():
            param.requires_grad = False
    elif scheme in {'full_finetune', 'no_pretrain'}:
        pass
    else:
        raise ValueError(f'Unknown transfer scheme: {scheme}')


def backbone_state_dict(model):
    return model.backbone.state_dict()


def heads_state_dict(model):
    return {'classification_head': model.classification_head.state_dict()}

model = StandardCNNTransformer(CONFIG).to(CONFIG['device'])
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
logger.info('Model instantiated. Total params=%d | Trainable params=%d', total_params, trainable_params)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
logger.info('CLIC-Net model definition and instantiation completed.')


class AdaptiveCounterfactualMask(nn.Module):
    def __init__(self, num_classes=4, num_leads=12):
        super().__init__()
        self.mask_logits = nn.Parameter(torch.zeros(num_classes, num_leads))

    def forward(self):
        return torch.sigmoid(self.mask_logits)

    def mask_for_targets(self, y):
        return self.forward()[y]


class AdaptiveCLICNet(StandardCNNTransformer):
    """StandardCNNTransformer with only a learnable class-specific counterfactual lead mask added."""
    def __init__(self, config):
        super().__init__(config)
        self.counterfactual_mask = AdaptiveCounterfactualMask(config['model']['main_outputs'], config['model']['input_leads'])

    def forward(self, x):
        out = super().forward(x)
        out['class_lead_mask'] = self.counterfactual_mask()
        return out

    def make_counterfactual(self, x, y):
        mask_y = self.counterfactual_mask.mask_for_targets(y).unsqueeze(1)
        return x * (1.0 - mask_y)


def mask_diversity_loss(class_lead_mask):
    normalized = F.normalize(class_lead_mask, p=2, dim=1)
    sim = normalized @ normalized.T
    n = sim.size(0)
    off_diag = sim[~torch.eye(n, dtype=torch.bool, device=sim.device)]
    return off_diag.mean()


def adaptive_counterfactual_loss(model, x, y, logits_original):
    x_cf = model.make_counterfactual(x, y)
    logits_cf = model(x_cf)['logits']
    batch_idx = torch.arange(y.size(0), device=y.device)
    delta = logits_original[batch_idx, y] - logits_cf[batch_idx, y]
    loss_sens = F.relu(CONFIG['counterfactual_margin'] - delta).mean()
    non_target = torch.ones_like(logits_original, dtype=torch.bool)
    non_target[batch_idx, y] = False
    loss_cons = F.mse_loss(logits_cf[non_target], logits_original.detach()[non_target])
    class_lead_mask = model.counterfactual_mask()
    loss_sparse = class_lead_mask.abs().mean()
    loss_div = mask_diversity_loss(class_lead_mask)
    return loss_sens, loss_cons, loss_sparse, loss_div


def total_loss_for_variant(model, x, y, variant, criterion):
    out = model(x)
    logits = out['logits']
    loss_cls = criterion(logits, y)
    if variant == 'clicnet_counterfactual':
        loss_sens, loss_cons, loss_sparse, loss_div = adaptive_counterfactual_loss(model, x, y, logits)
        loss = loss_cls + CONFIG['lambda_sens'] * loss_sens + CONFIG['lambda_cons'] * loss_cons + CONFIG['lambda_sparse'] * loss_sparse + CONFIG['lambda_div'] * loss_div
    else:
        z = torch.tensor(0.0, device=x.device)
        loss_sens = loss_cons = loss_sparse = loss_div = z
        loss = loss_cls
    return loss, out, {
        'loss_cls': float(loss_cls.detach().cpu()),
        'loss_sens': float(loss_sens.detach().cpu()),
        'loss_cons': float(loss_cons.detach().cpu()),
        'loss_sparse': float(loss_sparse.detach().cpu()),
        'loss_div': float(loss_div.detach().cpu()),
    }


def build_clicnet_model():
    return AdaptiveCLICNet(CONFIG).to(CONFIG['device'])


def save_learned_mask_artifacts(model, metrics_dir, plot_dir):
    if not hasattr(model, 'counterfactual_mask'):
        return
    mask = model.counterfactual_mask().detach().cpu().numpy()
    df = pd.DataFrame(mask, index=MAIN_LABEL_DISPLAY, columns=LEAD_ORDER)
    df.to_csv(metrics_dir / 'learned_class_lead_mask.csv')
    fig, ax = plt.subplots(figsize=(11, 4.8), dpi=400)
    sns.heatmap(df, annot=True, fmt='.3f', cmap='Oranges', linewidths=1, linecolor='black', ax=ax, annot_kws={'fontsize': 10})
    ax.set_xlabel('Lead')
    ax.set_ylabel('Class')
    ax.set_title('Learned Class-Specific Lead Mask', fontsize=16, weight='bold')
    fig.tight_layout()
    fig.savefig(plot_dir / 'learned_class_lead_mask_heatmap.png', bbox_inches='tight')
    plt.close(fig)


CLINICAL_LEAD_GROUPS = {
    'Normal': LEAD_ORDER,
    'Anterior': ['V1', 'V2', 'V3', 'V4'],
    'Inferior': ['II', 'III', 'aVF'],
    'Lateral': ['I', 'aVL', 'V5', 'V6'],
}


def save_learned_mask_clinical_overlap(model, metrics_dir, plot_dir):
    if not hasattr(model, 'counterfactual_mask'):
        return
    metrics_dir = Path(metrics_dir); plot_dir = Path(plot_dir)
    mask = model.counterfactual_mask().detach().cpu().numpy()
    rows = []
    overlap_matrix = []
    for class_idx, (internal_label, display_label) in enumerate(zip(MAIN_LABELS, MAIN_LABEL_DISPLAY)):
        values = mask[class_idx]
        order = np.argsort(values)[::-1]
        expected = set(CLINICAL_LEAD_GROUPS.get(internal_label, []))
        top3 = [LEAD_ORDER[i] for i in order[:3]]
        top5 = [LEAD_ORDER[i] for i in order[:5]]
        overlap_at_3 = len(set(top3) & expected) / max(1, min(3, len(expected)))
        overlap_at_5 = len(set(top5) & expected) / max(1, min(5, len(expected)))
        rows.append({
            'class': display_label,
            'internal_label': internal_label,
            'expected_clinical_leads': ','.join(CLINICAL_LEAD_GROUPS.get(internal_label, [])),
            'top3_learned_mask_leads': ','.join(top3),
            'top5_learned_mask_leads': ','.join(top5),
            'overlap_at_3': float(overlap_at_3),
            'overlap_at_5': float(overlap_at_5),
            'mean_expected_mask_weight': float(np.mean([values[LEAD_ORDER.index(lead)] for lead in expected])) if expected else np.nan,
            'mean_non_expected_mask_weight': float(np.mean([values[i] for i, lead in enumerate(LEAD_ORDER) if lead not in expected])) if expected and any(lead not in expected for lead in LEAD_ORDER) else np.nan,
        })
        overlap_matrix.append([overlap_at_3, overlap_at_5])
    df = pd.DataFrame(rows)
    df.to_csv(metrics_dir / 'learned_mask_clinical_overlap.csv', index=False)
    heat = pd.DataFrame(overlap_matrix, index=MAIN_LABEL_DISPLAY, columns=['overlap_at_3', 'overlap_at_5'])
    heat.to_csv(metrics_dir / 'learned_mask_clinical_overlap_matrix.csv')
    fig, ax = plt.subplots(figsize=(7.5, 4.8), dpi=400)
    sns.heatmap(heat, annot=True, fmt='.2f', vmin=0, vmax=1, cmap='Oranges', linewidths=1, linecolor='black', ax=ax, annot_kws={'fontsize': 12})
    ax.set_title('Learned Mask and Clinical Lead Overlap', fontsize=15, weight='bold')
    ax.set_xlabel('Overlap metric')
    ax.set_ylabel('Class')
    fig.tight_layout()
    fig.savefig(plot_dir / 'learned_mask_clinical_overlap_heatmap.png', bbox_inches='tight')
    fig.savefig(metrics_dir / 'learned_mask_clinical_overlap_heatmap.png', bbox_inches='tight')
    plt.close(fig)


def save_counterfactual_logit_drop_artifacts(model, loader, metrics_dir, plot_dir, split_name='test'):
    if not hasattr(model, 'counterfactual_mask'):
        return
    metrics_dir = Path(metrics_dir); plot_dir = Path(plot_dir)
    device = CONFIG.get('device', CONFIG.get('DEVICE', 'cpu'))
    model.eval()
    class_count = len(MAIN_LABELS)
    sums = np.zeros((class_count, class_count), dtype=np.float64)
    target_sums = np.zeros(class_count, dtype=np.float64)
    counts = np.zeros(class_count, dtype=np.float64)
    detail_rows = []
    with torch.no_grad():
        for batch in loader:
            x = batch['x'].to(device)
            y = batch['y'].to(device)
            logits_original = model(x)['logits']
            x_cf = model.make_counterfactual(x, y)
            logits_cf = model(x_cf)['logits']
            drop = (logits_original - logits_cf).detach().cpu().numpy()
            y_np = y.detach().cpu().numpy().astype(int)
            idx_np = batch['idx'].detach().cpu().numpy() if hasattr(batch['idx'], 'detach') else np.asarray(batch['idx'])
            for row_idx, true_class in enumerate(y_np):
                sums[true_class] += drop[row_idx]
                target_sums[true_class] += drop[row_idx, true_class]
                counts[true_class] += 1
                detail = {
                    'split': split_name,
                    'sample_idx': int(idx_np[row_idx]),
                    'true_class': MAIN_LABEL_DISPLAY[true_class],
                    'target_logit_drop': float(drop[row_idx, true_class]),
                }
                for class_idx, display_label in enumerate(MAIN_LABEL_DISPLAY):
                    detail[f'logit_drop_{display_label}'] = float(drop[row_idx, class_idx])
                detail_rows.append(detail)
    mean_drop = np.divide(sums, counts[:, None], out=np.zeros_like(sums), where=counts[:, None] > 0)
    target_mean = np.divide(target_sums, counts, out=np.zeros_like(target_sums), where=counts > 0)
    drop_df = pd.DataFrame(mean_drop, index=MAIN_LABEL_DISPLAY, columns=MAIN_LABEL_DISPLAY)
    drop_df.index.name = 'true_class_masked_by_learned_target_mask'
    drop_df.to_csv(metrics_dir / f'{split_name}_counterfactual_logit_drop_heatmap.csv')
    pd.DataFrame({
        'class': MAIN_LABEL_DISPLAY,
        'support': counts.astype(int),
        'mean_target_logit_drop': target_mean,
    }).to_csv(metrics_dir / f'{split_name}_counterfactual_target_logit_drop_by_class.csv', index=False)
    pd.DataFrame(detail_rows).to_csv(metrics_dir / f'{split_name}_counterfactual_logit_drop_details.csv', index=False)
    fig, ax = plt.subplots(figsize=(8.8, 6.8), dpi=400)
    sns.heatmap(drop_df, annot=True, fmt='.3f', cmap='Oranges', linewidths=1, linecolor='black', ax=ax, annot_kws={'fontsize': 11})
    ax.set_title(f'{split_name.title()} Counterfactual Logit Drop', fontsize=16, weight='bold')
    ax.set_xlabel('Logit class')
    ax.set_ylabel('True class using learned target mask')
    fig.tight_layout()
    fig.savefig(plot_dir / f'{split_name}_counterfactual_logit_drop_heatmap.png', bbox_inches='tight')
    fig.savefig(metrics_dir / f'{split_name}_counterfactual_logit_drop_heatmap.png', bbox_inches='tight')
    plt.close(fig)


## 7. Metrics


In [ ]:
def softmax_np(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def compute_classification_metrics(y_true, logits, label_names):
    prob = softmax_np(logits)
    pred = np.argmax(prob, axis=1)
    per_class_f1 = f1_score(y_true, pred, labels=np.arange(len(label_names)), average=None, zero_division=0)
    cm = confusion_matrix(y_true, pred, labels=np.arange(len(label_names)))
    metrics = {
        'accuracy': float((pred == y_true).mean()),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, pred)),
        'macro_f1': float(f1_score(y_true, pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, pred, average='weighted', zero_division=0)),
    }
    class_rows = []
    auc_values = []
    for idx, (label, display_label, f1_value) in enumerate(zip(label_names, MAIN_LABEL_DISPLAY, per_class_f1)):
        tp = float(cm[idx, idx])
        fn = float(cm[idx, :].sum() - cm[idx, idx])
        fp = float(cm[:, idx].sum() - cm[idx, idx])
        tn = float(cm.sum() - tp - fn - fp)
        sensitivity = tp / max(tp + fn, 1.0)
        specificity = tn / max(tn + fp, 1.0)
        try:
            auc_value = float(roc_auc_score((y_true == idx).astype(int), prob[:, idx]))
            auc_values.append(auc_value)
        except ValueError:
            auc_value = np.nan
        metrics[f'f1_{label}'] = float(f1_value)
        metrics[f'sensitivity_{label}'] = float(sensitivity)
        metrics[f'specificity_{label}'] = float(specificity)
        metrics[f'auc_{label}'] = float(auc_value) if np.isfinite(auc_value) else np.nan
        class_rows.append({
            'label': display_label,
            'internal_label': label,
            'f1_score': float(f1_value),
            'sensitivity': float(sensitivity),
            'specificity': float(specificity),
            'auc': float(auc_value) if np.isfinite(auc_value) else np.nan,
            'support': int((y_true == idx).sum()),
        })
    metrics['macro_auc'] = float(np.nanmean(auc_values)) if auc_values else np.nan
    return metrics, prob, pred, pd.DataFrame(class_rows)

def plot_training_curves(metrics_df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)
    axes[0].plot(metrics_df['epoch'], metrics_df['train_loss'], label='train')
    axes[0].plot(metrics_df['epoch'], metrics_df['val_loss'], label='val')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(metrics_df['epoch'], metrics_df['train_macro_f1'], label='train')
    axes[1].plot(metrics_df['epoch'], metrics_df['val_macro_f1'], label='val')
    axes[1].set_title('Macro F1')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / 'training_curve.png', bbox_inches='tight')
    fig.savefig(METRICS_DIR / 'training_curve.png', bbox_inches='tight')
    plt.close(fig)


def save_confusion_matrix(y_true, y_pred, labels, output_prefix):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(labels)))
    pd.DataFrame(cm, index=labels, columns=labels).to_csv(METRICS_DIR / f'{output_prefix}.csv')
    fig, ax = plt.subplots(figsize=(9, 8), dpi=400)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=labels, yticklabels=labels, ax=ax, cbar=True, linewidths=1.0, linecolor='black', annot_kws={'fontsize': 14})
    ax.set_xlabel('Predicted', fontsize=14)
    ax.set_ylabel('True', fontsize=14)
    ax.set_title(output_prefix.replace('_', ' ').title(), fontsize=16, weight='bold')
    fig.tight_layout()
    fig.savefig(METRICS_DIR / f'{output_prefix}.png', bbox_inches='tight')
    plt.close(fig)
    return cm


def save_roc_curves(y_true, prob, output_prefix):
    rows = []
    fig, ax = plt.subplots(figsize=(9, 7), dpi=400)
    for idx, label in enumerate(MAIN_LABEL_DISPLAY):
        binary_true = (y_true == idx).astype(int)
        if binary_true.min() == binary_true.max():
            continue
        fpr, tpr, _ = roc_curve(binary_true, prob[:, idx])
        auc_value = roc_auc_score(binary_true, prob[:, idx])
        ax.plot(fpr, tpr, linewidth=2.2, label=f'{label} AUC={auc_value:.3f}')
        for x, y in zip(fpr, tpr):
            rows.append({'label': label, 'fpr': float(x), 'tpr': float(y), 'auc': float(auc_value)})
    ax.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1.2)
    ax.set_xlabel('False Positive Rate', fontsize=13)
    ax.set_ylabel('True Positive Rate / Sensitivity', fontsize=13)
    ax.set_title(output_prefix.replace('_', ' ').title(), fontsize=16, weight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'{output_prefix}.png', bbox_inches='tight')
    fig.savefig(METRICS_DIR / f'{output_prefix}.png', bbox_inches='tight')
    plt.close(fig)
    pd.DataFrame(rows).to_csv(METRICS_DIR / f'{output_prefix}.csv', index=False)


def save_pr_curves(y_true, prob, output_prefix):
    rows = []
    fig, ax = plt.subplots(figsize=(9, 7), dpi=400)
    for idx, label in enumerate(MAIN_LABEL_DISPLAY):
        binary_true = (y_true == idx).astype(int)
        if binary_true.min() == binary_true.max():
            continue
        precision, recall, _ = precision_recall_curve(binary_true, prob[:, idx])
        auprc_value = average_precision_score(binary_true, prob[:, idx])
        ax.plot(recall, precision, linewidth=2.2, label=f'{label} AUPRC={auprc_value:.3f}')
        for r, p in zip(recall, precision):
            rows.append({'label': label, 'recall': float(r), 'precision': float(p), 'auprc': float(auprc_value)})
    ax.set_xlabel('Recall / Sensitivity', fontsize=13)
    ax.set_ylabel('Precision', fontsize=13)
    ax.set_title(output_prefix.replace('_', ' ').title(), fontsize=16, weight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'{output_prefix}.png', bbox_inches='tight')
    fig.savefig(METRICS_DIR / f'{output_prefix}.png', bbox_inches='tight')
    plt.close(fig)
    pd.DataFrame(rows).to_csv(METRICS_DIR / f'{output_prefix}.csv', index=False)


## 8. Variant training, evaluation, and transfer-ready export


In [ ]:
def run_epoch_variant(model, loader, criterion, variant, optimizer=None, device='cpu'):
    training=optimizer is not None; model.train(training)
    total_loss=0.0; n=0; sums={k:0.0 for k in ['loss_cls','loss_sens','loss_cons','loss_sparse','loss_div']}; logits_all=[]; y_all=[]; idx_all=[]
    context=torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in loader:
            x=batch['x'].to(device); y=batch['y'].to(device)
            if training: optimizer.zero_grad(set_to_none=True)
            loss, outputs, parts=total_loss_for_variant(model,x,y,variant,criterion)
            if training:
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0); optimizer.step()
            bs=x.size(0); total_loss += float(loss.detach().cpu())*bs; n += bs
            for k in sums: sums[k] += parts[k]*bs
            logits_all.append(outputs['logits'].detach().cpu().numpy()); y_all.append(y.detach().cpu().numpy()); idx_all.append(batch['idx'].detach().cpu().numpy())
    state={'loss':total_loss/max(n,1),'logits':np.concatenate(logits_all),'y_true':np.concatenate(y_all),'idx':np.concatenate(idx_all)}
    for k,v in sums.items(): state[k]=v/max(n,1)
    return state

def save_predictions(path, labels_df, y_true, prob, pred):
    df = labels_df.copy().reset_index(drop=True)
    df['true_class_index'] = y_true.astype(int)
    df['pred_class_index'] = pred.astype(int)
    df['true_label'] = [MAIN_LABELS[int(i)] for i in y_true]
    df['pred_label'] = [MAIN_LABELS[int(i)] for i in pred]
    for i, name in enumerate(MAIN_LABELS):
        df[f'prob_{name}'] = prob[:, i]
    df.to_csv(path, index=False)
    return df

def variant_heads_state_dict(model):
    out={'classification_head': model.classification_head.state_dict()}
    if hasattr(model,'counterfactual_mask'): out['counterfactual_mask']=model.counterfactual_mask.state_dict()
    return out

def save_variant_transfer_ready(model, variant, variant_dir):
    transfer_dir=variant_dir/'transfer_ready'; transfer_dir.mkdir(parents=True, exist_ok=True)
    np.save(transfer_dir/'normalizer_mean.npy', normalizer.mean_.astype(np.float32)); np.save(transfer_dir/'normalizer_std.npy', normalizer.std_.astype(np.float32))
    label_mapping=dict(LABEL_MAPPING); model_config={**CONFIG['model'],'model_name':'CLIC-Net','baseline_variant':CONFIG['baseline_variant'],'pooling_type':CONFIG['pooling_type'],'task_type':LABEL_MAPPING['task_type'],'classification_head':CONFIG['classification_head'],'variant':variant}
    torch.save({'backbone_state_dict':model.backbone.state_dict(),'config':CONFIG,'label_mapping':label_mapping,'lead_order':LEAD_ORDER,'normalizer_mean_path':'normalizer_mean.npy','normalizer_std_path':'normalizer_std.npy','baseline_variant':CONFIG['baseline_variant'],'pooling_type':CONFIG['pooling_type'],'variant':variant}, transfer_dir/'clicnet_backbone.pt')
    torch.save({'model_state_dict':model.state_dict(),'backbone_state_dict':model.backbone.state_dict(),'heads_state_dict':variant_heads_state_dict(model),'config':CONFIG,'label_mapping':label_mapping,'lead_order':LEAD_ORDER,'main_label_names':MAIN_LABELS,'main_label_display':MAIN_LABEL_DISPLAY,'baseline_variant':CONFIG['baseline_variant'],'pooling_type':CONFIG['pooling_type'],'variant':variant}, transfer_dir/'clicnet_full_model.pt')
    torch.save({'heads_state_dict':variant_heads_state_dict(model),'config':CONFIG,'label_mapping':label_mapping,'variant':variant}, transfer_dir/'clicnet_heads.pt')
    with open(transfer_dir/'label_mapping.json','w') as f: json.dump(label_mapping,f,indent=2)
    with open(transfer_dir/'model_config.json','w') as f: json.dump(model_config,f,indent=2)
    with open(transfer_dir/'clicnet_architecture_config.json','w') as f: json.dump(BASELINE_ARCHITECTURE_CONFIG,f,indent=2)


variant = CONFIG['variant']
logger.info('CLIC-Net pretraining started. Variant: %s', variant)
model = AdaptiveCLICNet(CONFIG).to(CONFIG['device'])
criterion = nn.CrossEntropyLoss().to(CONFIG['device'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=CONFIG['scheduler_patience'])

def save_checkpoint(path, epoch, metric, val_loss):
    torch.save({
        'model_state_dict': model.state_dict(),
        'backbone_state_dict': model.backbone.state_dict(),
        'heads_state_dict': variant_heads_state_dict(model),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch,
        'best_metric': metric,
        'best_val_macro_f1': metric,
        'val_loss': val_loss,
        'config': CONFIG,
        'label_mapping': {'main_labels': MAIN_LABELS, 'main_label_display': MAIN_LABEL_DISPLAY},
        'lead_order': LEAD_ORDER,
        'main_label_names': MAIN_LABELS,
        'variant': variant,
        'task_type': 'single_label_4_class_softmax_counterfactual',
    }, path)

rows = []
best_macro_f1 = -np.inf
best_val_loss = np.inf
for epoch in range(1, CONFIG['epochs'] + 1):
    tr = run_epoch_variant(model, train_loader, criterion, variant, optimizer, CONFIG['device'])
    va = run_epoch_variant(model, val_loader, criterion, variant, None, CONFIG['device'])
    tm, _, _, _ = compute_classification_metrics(tr['y_true'], tr['logits'], MAIN_LABELS)
    vm, _, _, _ = compute_classification_metrics(va['y_true'], va['logits'], MAIN_LABELS)
    scheduler.step(vm['macro_f1'])
    row = {
        'variant': variant,
        'epoch': epoch,
        'train_loss': tr['loss'],
        'val_loss': va['loss'],
        'train_loss_cls': tr['loss_cls'],
        'val_loss_cls': va['loss_cls'],
        'train_loss_sens': tr['loss_sens'],
        'val_loss_sens': va['loss_sens'],
        'train_loss_cons': tr['loss_cons'],
        'val_loss_cons': va['loss_cons'],
        'train_loss_sparse': tr['loss_sparse'],
        'val_loss_sparse': va['loss_sparse'],
        'train_loss_div': tr['loss_div'],
        'val_loss_div': va['loss_div'],
        'train_macro_f1': tm['macro_f1'],
        'val_macro_f1': vm['macro_f1'],
        'learning_rate': optimizer.param_groups[0]['lr'],
    }
    for label in MAIN_LABELS:
        row[f'train_f1_{label}'] = tm[f'f1_{label}']
        row[f'val_f1_{label}'] = vm[f'f1_{label}']
    rows.append(row)
    pd.DataFrame(rows).to_csv(LOG_DIR / 'metrics.csv', index=False)
    pd.DataFrame(rows).to_csv(METRICS_DIR / 'metrics.csv', index=False)
    logger.info('Epoch %03d/%03d train_loss=%.5f val_loss=%.5f val_macro_f1=%.4f', epoch, CONFIG['epochs'], tr['loss'], va['loss'], vm['macro_f1'])
    save_checkpoint(CKPT_DIR / 'last.pt', epoch, best_macro_f1, va['loss'])
    if vm['macro_f1'] > best_macro_f1:
        best_macro_f1 = vm['macro_f1']
        save_checkpoint(CKPT_DIR / 'best_macro_f1.pt', epoch, best_macro_f1, va['loss'])
    if va['loss'] < best_val_loss:
        best_val_loss = va['loss']
        save_checkpoint(CKPT_DIR / 'best_val_loss.pt', epoch, best_macro_f1, va['loss'])
    if epoch >= 5 and epoch % 5 == 0:
        save_checkpoint(PER_5_EPOCH_CKPT_DIR / f'epoch_{epoch:03d}.pt', epoch, best_macro_f1, va['loss'])

ckpt = torch.load(CKPT_DIR / 'best_macro_f1.pt', map_location=CONFIG['device'])
model.load_state_dict(ckpt['model_state_dict'])
val_state = run_epoch_variant(model, val_loader, criterion, variant, None, CONFIG['device'])
test_state = run_epoch_variant(model, test_loader, criterion, variant, None, CONFIG['device'])
val_metrics, val_prob, val_pred, val_class_metrics = compute_classification_metrics(val_state['y_true'], val_state['logits'], MAIN_LABELS)
test_metrics, test_prob, test_pred, test_class_metrics = compute_classification_metrics(test_state['y_true'], test_state['logits'], MAIN_LABELS)
final_metrics_df = pd.DataFrame([
    {'split': 'val', **val_metrics, 'loss': val_state['loss']},
    {'split': 'test', **test_metrics, 'loss': test_state['loss']},
])
final_metrics_df.to_csv(LOG_DIR / 'final_metrics.csv', index=False)
final_metrics_df.to_csv(METRICS_DIR / 'final_metrics.csv', index=False)
val_predictions_df = save_predictions(PRED_DIR / 'val_predictions.csv', labels_val.iloc[val_state['idx']], val_state['y_true'], val_prob, val_pred)
test_predictions_df = save_predictions(PRED_DIR / 'test_predictions.csv', labels_test.iloc[test_state['idx']], test_state['y_true'], test_prob, test_pred)
save_confusion_matrix(val_state['y_true'], val_pred, MAIN_LABEL_DISPLAY, 'val_confusion_matrix')
save_confusion_matrix(test_state['y_true'], test_pred, MAIN_LABEL_DISPLAY, 'test_confusion_matrix')
test_f1_rows = []
for label, display_label in zip(MAIN_LABELS, MAIN_LABEL_DISPLAY):
    test_f1_rows.append({'split': 'test', 'label': display_label, 'internal_label': label, 'f1_score': test_metrics[f'f1_{label}']})
test_f1_df = pd.DataFrame(test_f1_rows)
test_f1_df.to_csv(PLOT_DIR / 'test_f1_scores_by_class.csv', index=False)
test_f1_df.to_csv(METRICS_DIR / 'test_f1_scores_by_class.csv', index=False)
val_class_metrics.insert(0, 'split', 'val')
test_class_metrics.insert(0, 'split', 'test')
pd.concat([val_class_metrics, test_class_metrics], ignore_index=True).to_csv(METRICS_DIR / 'per_class_metrics.csv', index=False)
pd.concat([val_class_metrics, test_class_metrics], ignore_index=True).to_csv(METRICS_DIR / 'classification_report.csv', index=False)
save_roc_curves(val_state['y_true'], val_prob, 'val_roc_curve')
save_roc_curves(test_state['y_true'], test_prob, 'test_roc_curve')
save_pr_curves(val_state['y_true'], val_prob, 'val_pr_curve')
save_pr_curves(test_state['y_true'], test_prob, 'test_pr_curve')
f1_summary_df = pd.DataFrame([
    {'split': 'val', 'macro_f1': val_metrics['macro_f1'], 'weighted_f1': val_metrics['weighted_f1'], 'balanced_accuracy': val_metrics['balanced_accuracy'], 'macro_auc': val_metrics['macro_auc'], 'accuracy': val_metrics['accuracy']},
    {'split': 'test', 'macro_f1': test_metrics['macro_f1'], 'weighted_f1': test_metrics['weighted_f1'], 'balanced_accuracy': test_metrics['balanced_accuracy'], 'macro_auc': test_metrics['macro_auc'], 'accuracy': test_metrics['accuracy']},
])
f1_summary_df.to_csv(PLOT_DIR / 'f1_score_summary.csv', index=False)
f1_summary_df.to_csv(METRICS_DIR / 'f1_score_summary.csv', index=False)
plot_training_curves(pd.read_csv(METRICS_DIR / 'metrics.csv'))
save_learned_mask_artifacts(model, METRICS_DIR, PLOT_DIR)
save_learned_mask_clinical_overlap(model, METRICS_DIR, PLOT_DIR)
save_counterfactual_logit_drop_artifacts(model, test_loader, METRICS_DIR, PLOT_DIR, split_name='test')
save_variant_transfer_ready(model, variant, OUTPUT_DIR)
summary = pd.DataFrame([{
    'variant': variant,
    'status': 'OK',
    'test_accuracy': test_metrics['accuracy'],
    'test_macro_f1': test_metrics['macro_f1'],
    'test_weighted_f1': test_metrics['weighted_f1'],
    'test_f1_NORM': test_metrics['f1_Normal'],
    'test_f1_AMI': test_metrics['f1_Anterior'],
    'test_f1_IMI': test_metrics['f1_Inferior'],
    'test_f1_Lateral_involved_MI': test_metrics['f1_Lateral'],
    'checkpoint_path': str(CKPT_DIR / 'best_macro_f1.pt'),
}])
summary.to_csv(ROOT_METRICS_DIR / 'pretrain_clicnet_summary.csv', index=False)
display(summary)
